# Full-epoch Poly-1 companion pilot — ResNet18 on KolektorSDD2

**Paired with the completed 420-image CE pilot.** This notebook creates a *separate* Poly-1 run directory and never overwrites the verified CE pilot.

**New independent experiment:** 420 training images, Poly-1 loss, seed 20261028, **30 true complete epochs = 3,150 optimizer updates**. Validation: 180 image holdout from official **TRAIN** only (150 normal + 30 defective), no official TEST images are used. Model selected by internal validation Macro-F1. Historical 216-step results and all 24 existing evaluations are left unchanged.

**Before running:** Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Authorize your own Google Drive mount when prompted. The original source data already lives in `MyDrive/Nazih_Coursework/05_Official_Dataset/KolektorSDD2.zip`. The code automatically clones the existing GitHub branch, reconstructs and verifies the old frozen sampling manifest if absent, and saves incremental progress/checkpoints in **MyDrive/Nazih_Coursework/06_Full_Epoch_Pilot**.

If Colab disconnects after an epoch checkpoint, open this notebook and **Run all** again: the pilot resumes from the last completed epoch. Any old results, checkpoints, original protocol and original Drive folders are never rewritten. New results are exploratory until reviewed; test-set claims are not generated.


In [ ]:
import torch
import torchvision
assert torch.cuda.is_available(), "GPU unavailable: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil, hashlib, json
REPOSITORY = "https://github.com/ID9999999999999/DL-Computer-vision.git"
BRANCH = "nazih-training-set-size-20261109"
REPO = Path("/content/nazih_full_epoch_project")
PROJECT = REPO / "COURSEWORK_NAZIH"
DRIVE_BASE = Path("/content/drive/MyDrive/Nazih_Coursework")
DRIVE_OUT = DRIVE_BASE / "06_Full_Epoch_Pilot"
DRIVE_DATASET = DRIVE_BASE / "05_Official_Dataset" / "KolektorSDD2.zip"

assert DRIVE_BASE.is_dir(), f"Existing project Drive folder not visible: {DRIVE_BASE}"
assert DRIVE_OUT.is_dir(), f"Expected backup folder missing (created earlier): {DRIVE_OUT}"
assert DRIVE_DATASET.is_file(), f"Official ZIP missing on Drive: {DRIVE_DATASET}"

if not (REPO / ".git").is_dir():
    assert not REPO.exists(), "Refusing to overwrite unrelated /content project"
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH,
                    "--depth", "1", REPOSITORY, str(REPO)], check=True)
else:
    status = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip()
    assert not status, "Unexpected local changes in Colab clone"
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)

assert (PROJECT / "scripts" / "full_epoch_pilot.py").is_file()
assert (PROJECT / "protocol.json").is_file()
output_link = PROJECT / "data" / "full_epoch_pilot"
output_link.parent.mkdir(parents=True, exist_ok=True)
if output_link.is_symlink():
    assert output_link.resolve() == DRIVE_OUT.resolve(), "Output points to the wrong Drive folder!"
elif output_link.exists():
    raise RuntimeError("Refusing to overwrite existing non-symlink output directory")
else:
    output_link.symlink_to(DRIVE_OUT, target_is_directory=True)

print("✅ Code cloned and persistent Drive output linked")
print("✅ Existing historical results remain untouched")
print("Output:", output_link.resolve())

In [ ]:
EXPECTED_SHA256 = "edcdb486809b24f1d17b785e30c52fafc5999554dd5fe18ddf77b61ceb6f36a8"
LOCAL_DATASET = Path("/content/KolektorSDD2.zip")
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for piece in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(piece)
    return h.hexdigest()

if not LOCAL_DATASET.exists():
    free = shutil.disk_usage("/content").free
    assert free > DRIVE_DATASET.stat().st_size + 1024**3, "Not enough Colab disk space"
    shutil.copy2(DRIVE_DATASET, LOCAL_DATASET)

assert sha256(LOCAL_DATASET) == EXPECTED_SHA256, "Official dataset SHA256 differs. Stop."
print("✅ Dataset copied from Google Drive and SHA256 verified")
print("ZIP bytes:", LOCAL_DATASET.stat().st_size)

In [ ]:
SCRIPT = PROJECT / "scripts" / "full_epoch_pilot.py"
PILOT_ARGS = [
    "--archive", str(LOCAL_DATASET),
    "--size", "420", "--seed", "20261028",
    "--loss", "poly1", "--epochs", "30",
    "--batch-size", "4",
]
subprocess.run([sys.executable, str(SCRIPT), *PILOT_ARGS, "--check-only"],
               cwd=PROJECT, check=True)
print("✅ PRE-FLIGHT SUCCESS: frozen training set, disjoint train-only validation, no official test images opened")

## Run the 30-epoch pilot

This executes real backpropagation on the GPU. The **best model**, the **resume state**, the fixed validation split and each epoch's **Train/Validation** metrics are saved to your Drive folder. **Do not run another copy of the same pilot simultaneously**, or the output files would conflict.


In [ ]:
subprocess.run([
    sys.executable, str(SCRIPT), *PILOT_ARGS, "--device", "cuda"
], cwd=PROJECT, check=True)
print("✅ 30 TRUE FULL EPOCHS TRAINED")

In [ ]:
from pathlib import Path
RESULT_DIR = DRIVE_OUT / "n420_seed20261028_poly1"
complete_path = RESULT_DIR / "COMPLETE.json"
assert complete_path.is_file(), "Training not yet complete; rerun Run all to resume."
result = json.loads(complete_path.read_text(encoding="utf-8"))
history = result["history"]
assert len(history) == 30 and [x["full_epoch"] for x in history] == list(range(1, 31))
assert all(x["training_optimizer_steps"] == 105 for x in history)
assert result["identity"]["total_planned_optimizer_steps"] == 3150
assert result["official_test_evaluated"] is False
assert sha256(RESULT_DIR / "best_model.pt") == result["best_model_sha256"]
print("✅ TRAINING VERIFIED — 30 actual full epochs × 420 images")
print("✅ Optimizer steps:", sum(x["training_optimizer_steps"] for x in history))
print("✅ Best epoch selected exclusively by train-only validation:", result["best_epoch"])
print("✅ Best validation Macro-F1:", result["best_validation_macro_f1"])
print("✅ Original 24 evaluations untouched")
print("✅ Persistent backup:", RESULT_DIR)

In [ ]:
import matplotlib.pyplot as plt
epochs = [h["full_epoch"] for h in history]
train_loss = [h["train_objective"] for h in history]
val_loss = [h["validation_objective"] for h in history]
val_macro_f1 = [h["validation"]["macro_f1"] for h in history]

plt.figure(figsize=(9, 4))
plt.plot(epochs, train_loss, label="Training objective")
plt.plot(epochs, val_loss, label="Validation objective")
plt.xlabel("Full dataset epochs")
plt.ylabel("Objective (Poly-1)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(epochs, val_macro_f1, label="Validation Macro-F1")
plt.axvline(result["best_epoch"], linestyle="--", label="Selected best epoch")
plt.xlabel("Full dataset epochs")
plt.ylabel("Validation Macro-F1")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
print("No official TEST images were used in the new pilot.")